In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.utils.data as data_utils
import numpy as np
import os
import torch
import torch.nn as nn
torch.backends.cudnn.enabled=False
import torch.optim as optim
from torch.utils.data import Dataset
import pickle
import joblib
from sklearn.metrics import mean_absolute_error 
from lifelines.utils import concordance_index
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import subprocess
import argparse
import random
from sklearn.model_selection import KFold
import ast
from torch.utils.data import WeightedRandomSampler

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
# Example usage

def random_train_val_split(X_combined, y_combined, n_train, n_val):
    n_total = n_train + n_val
    idx = np.random.permutation(n_total)

    train_idx = idx[:n_train]
    val_idx = idx[n_train:]

    X_train = X_combined[train_idx]
    y_train = y_combined[train_idx]

    X_val = X_combined[val_idx]
    y_val = y_combined[val_idx]

    return X_train, y_train, X_val, y_val



# Example usag
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

class CurrDataset(Dataset):
    def __init__(self, name, data):
        super().__init__()
        self.name = name
        self.features = data['features']
        self.labels = data['labels']
        self.length = len(self.features)
        
    def __len__(self):
        """Return number of sequences."""
        return self.length

    def __getitem__(self, index):
        """Return sequence and label at index."""
        return self.features[index].float(), self.labels[index].long()


class DeepHit(nn.Module):
    def __init__(self, num_input, num_output, num_layers, hidden_size):
        super(DeepHit, self).__init__()
        self.num_output = num_output
        if num_layers == 1:
            self.feature_extractor = nn.Sequential(
                nn.Linear(num_input, num_output)
            )
        elif num_layers == 2:
            self.feature_extractor = nn.Sequential(
                nn.Linear(num_input, hidden_size),
                nn.ReLU(),
                nn.Linear(hidden_size, num_output)
            )
        elif num_layers == 3:
            self.feature_extractor = nn.Sequential(
                nn.Linear(num_input, hidden_size),
                nn.ReLU(),
                nn.Linear(hidden_size, hidden_size),
                nn.ReLU(),
                nn.Linear(hidden_size, num_output)
            )
                
    def forward(self, x, event_times):
        logits = self.feature_extractor(x)
        
        predictions = F.softmax(logits, dim=1)
        loss = self.total_loss(logits, event_times).mean()
                
        return torch.argmax(predictions, 1), loss
    
    def total_loss(self, logits, event_times):
        event_times = event_times.long()
        batch_size, num_times = logits.shape
        event_times = event_times.clamp(0, num_times - 1)
                
        predictions = F.softmax(logits, dim=1)
        log_probs = torch.log(predictions + 1e-8)

        event_loss = -log_probs[torch.arange(batch_size), event_times]        
        return event_loss

params = []
hospital_types = ['low_resource', 'moderate_resource', 'high_resource']
strategies = ['covariate']
for strategy in strategies:
    if strategy == 'hybrid':
        params.extend(['%s_T%d_%s'%(strategy, t, h) for t in thresholds for h in hospital_types])
    elif strategy == 'outcome':
        params.extend(['%s_T%d'%(strategy, t) for t in thresholds])
    elif strategy == 'covariate':
        params.extend(['%s_%s'%(strategy, h) for h in hospital_types])
    
# Load and parse each line
file_path = 'oracle.txt'
# Load and parse each line
hyps = []
with open(file_path, 'r') as f:
    for line in f:
        if len(line.split(',')) > 2:
            hyps.append(ast.literal_eval(line.strip()))

print(params)
suffixcount = -1
for suffix in params:
    suffixcount += 1
    X_test = joblib.load('$CWITE_DATA_ROOT/oracle_train_support50_propbin_data/X_test_%s.joblib'%suffix)
    y_test = joblib.load('$CWITE_DATA_ROOT/oracle_train_support50_propbin_data/orig_y_test_%s.joblib'%suffix)

    X_train = joblib.load('$CWITE_DATA_ROOT/oracle_train_support50_propbin_data/X_train_%s.joblib'%suffix)
    y_train = joblib.load('$CWITE_DATA_ROOT/oracle_train_support50_propbin_data/orig_y_train_%s.joblib'%suffix)

    X_val = joblib.load('$CWITE_DATA_ROOT/oracle_train_support50_propbin_data/X_val_%s.joblib'%suffix)
    y_val = joblib.load('$CWITE_DATA_ROOT/oracle_train_support50_propbin_data/orig_y_val_%s.joblib'%suffix)

    # Compute class counts
    labels_np = y_train  # numpy array
    class_sample_count = np.array([np.sum(labels_np == t) for t in np.unique(labels_np)])
    
    # Weight for each sample
    weights = 1. / class_sample_count
    samples_weight = np.array([weights[t] for t in labels_np])
    
    samples_weight = torch.from_numpy(samples_weight).double()
    sampler = WeightedRandomSampler(samples_weight, len(samples_weight))


    
    f_num_layers, f_hidden_size, f_batch_size, f_lr, f_wd = hyps[suffixcount]


    train_dataset = CurrDataset("train", {'features':torch.tensor(X_train), 'labels':torch.tensor(y_train)})
    train_loader = data_utils.DataLoader(train_dataset,
                                         batch_size=f_batch_size,
                                         sampler=sampler)


    val_dataset = CurrDataset("train", {'features':torch.tensor(X_val), 'labels':torch.tensor(y_val)})
    val_loader = data_utils.DataLoader(val_dataset,
                                         batch_size=f_batch_size,
                                         shuffle=False)

    test_dataset = CurrDataset("train", {'features':torch.tensor(X_test), 'labels':torch.tensor(y_test)})
    test_loader = data_utils.DataLoader(test_dataset,
                                         batch_size=f_batch_size,
                                         shuffle=False)


    model = DeepHit(X_train.shape[1], max(y_train), f_num_layers, f_hidden_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=f_lr, weight_decay=f_wd)
    optimizer.zero_grad()

    val_losses = []
    train_losses = []
    stop = -1
    for epoch in range(500):
        curr_train_losses = []
        for batch_idx, (data, label) in enumerate(train_loader):
            optimizer.zero_grad()
            data, label = data.to(device), label.to(device).float().to(device) 
            _, loss = model(data, label)
            loss.backward()
            curr_train_losses.append(loss.detach().cpu().numpy())
            # step
            optimizer.step()

        train_losses.append(np.mean(curr_train_losses))
        curr_val_losses = []
        for batch_idx, (data, label) in enumerate(val_loader):
            data, label = data.to(device), label.to(device).float().to(device)     
            output, loss = model(data, label)
            curr_val_losses.append(loss.detach().cpu().numpy())
        val_losses.append(np.mean(curr_val_losses))
        torch.save(model, '$CWITE_DATA_ROOT/support50_propbin_models/oracle_epoch%d'%(epoch))


        if val_losses[-1] > min(val_losses):
            stop += 1
        if val_losses[-1] == min(val_losses):
            stop = 0
        if stop == 15:
            break
            
    best_epoch = np.argmin(val_losses)
    model = DeepHit(X_train.shape[1], max(y_train), f_num_layers, f_hidden_size).to(device)
    model = torch.load('$CWITE_DATA_ROOT/support50_propbin_models/oracle_epoch%d'%(best_epoch))
    torch.save(model, '$CWITE_DATA_ROOT/support50_propbin_oracle/best_oracle_%s'%(suffix))

    folder_path = "$CWITE_DATA_ROOT/support50_propbin_models"
    os.system(f"rm -f {folder_path}/*oracle**")


cuda
['covariate_low_resource', 'covariate_moderate_resource', 'covariate_high_resource']


/tmp/ipykernel_965090/197479364.py:222: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load('$CWITE_DATA_ROOT/support50_propbin_models/oracle_epoch%d'%(best_epo